In [1]:
# conda env 'at3d_pure_k9'
from pathlib import Path
PROJECT_ROOT = Path().resolve().parent

import at3d
from collections import OrderedDict
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

In [2]:
fm_file = PROJECT_ROOT / 'experiments' / 'CAMP2Ex_RF17' / 'V04' / 'default' / 'iterations' / 'iter021.nc'

save_LUT = True
LUT_outfile = PROJECT_ROOT / 'NK1990_1D_Retrieval' / 'LUT_V04.nc'

sensors, solvers, rte_grid = at3d.util.load_forward_model(fm_file)

wavelength = sensors.get_unique_solvers()
assert len(wavelength == 1)
wavelength = wavelength[0]

## Create rte_grid with same vertical grid, but only a 3 x 3 in the horizontal
rte_grid = at3d.grid.make_grid(rte_grid.delx.data, 3, rte_grid.dely.data, 3, rte_grid.z.data)

## Define cloud base and cloud top
cb = 0.4
ct = 0.7
i_low = np.argwhere(rte_grid.z.data >= cb)[0][0]
i_up = np.argwhere(rte_grid.z.data <= ct)[-1][0]
thickness = rte_grid.z.data[i_up] - rte_grid.z.data[i_low]
## Define optical depths in LUT
tausA = np.linspace(0, 4.75, 20)
tausB = np.logspace(np.log10(5), 2, 40)
taus = np.append(tausA, tausB)

/data/keeling/a/johntl3/miniconda3/envs/at3d_pure_k9/lib/python3.10/site-packages/at3d/solver.py:2337: UserWarning: Number of SHDOM grid cells with optical depth greater than 2: '51'. Max cell optical depth: '3.0740342140197754'
  warnings.warn("Number of SHDOM grid cells with optical depth greater than 2: '{}'. "


In [3]:
## Make a sensor to observe radiance
LUT_sensors = at3d.containers.SensorsDict()
LUT_sensor = at3d.sensor.orthographic_projection(
    wavelength=wavelength,
    bounding_box=rte_grid,
    x_resolution=rte_grid.delx.data,
    y_resolution=rte_grid.dely.data,
    azimuth=0,
    zenith=0,
    altitude='TOA',
    stokes='I',
    sub_pixel_ray_args={'method': None}
)
LUT_sensors.add_sensor('LUT_sensor', LUT_sensor)

In [4]:
## Set up LUT solver with the features of the retreival solver

# Numerical parameters
numerical_parameters = solvers[wavelength].numerical_params
numerical_parameters['ip_flag'] = 3 # Set to independent pixel approximation
numerical_parameters['split_accuracy'] = 1e-3
# Surface
surface = solvers[wavelength].surface

# Source
source = solvers[wavelength].source

# Medium
medium = OrderedDict()
medium['background'] = solvers[wavelength].medium['background'].isel(x=slice(0,3), y=slice(0,3))
medium['cloud'] = solvers[wavelength].medium['cloud'].isel(x=slice(0,3), y=slice(0,3))

In [5]:
Is = np.zeros(taus.shape)
num_unique = np.zeros(taus.shape)
diff = np.zeros(taus.shape)
LUT_solvers = at3d.containers.SolversDict()
for i, tau in enumerate(taus):
    ext = np.zeros((rte_grid.dims['x'], rte_grid.dims['y'], rte_grid.dims['z']))
    ext[:, :, i_low:i_up] = tau/thickness
    medium['cloud']['extinction'] = (['x','y','z'], ext)

    LUT_solver = at3d.solver.RTE(
        numerical_params = numerical_parameters,
        medium = medium,
        source = source,
        surface = surface,
        num_stokes = 1
    )
    LUT_solvers.add_solver(wavelength, LUT_solver)
    LUT_sensors.get_measurements(LUT_solvers)

    cam = LUT_sensors['LUT_sensor']['sensor_list'][0]
    num_unique[i] = len(np.unique(cam.I.data))
    diff[i] = np.max(cam.I.data) - np.min(cam.I.data)
    # assert num_unique[i] <= 3, "Tau = {}\n{}".format(tau, cam.I)
    Is[i] = np.mean(cam.I.data)

  ! Iter Log(Sol)  SplitCrit  Npoints  Nsh(avg)   [Radiance 0.670 micron]
     1  -1.013  0.384E-01      135   256.00  1.000   [Radiance 0.670 micron]
     2  -1.470  0.103E-01      144   256.00  1.000   [Radiance 0.670 micron]
     3  -1.865  0.102E-01      144   256.00  1.000   [Radiance 0.670 micron]
     4  -2.238  0.102E-01      144   256.00  1.000   [Radiance 0.670 micron]
     5  -2.824  0.267E-02      171   256.00  1.000   [Radiance 0.670 micron]
     6  -2.673  0.675E-03      216   256.00  1.000   [Radiance 0.670 micron]
     7  -2.986  0.663E-03      216   256.00  1.000   [Radiance 0.670 micron]
     8  -3.325  0.665E-03      216   256.00  1.000   [Radiance 0.670 micron]
     9  -4.091  0.666E-03      216   256.00  1.000   [Radiance 0.670 micron]
 ! Iterations:      9     Final Criterion:  0.000081   [Radiance 0.670 micron]
  ! Iter Log(Sol)  SplitCrit  Npoints  Nsh(avg)   [Radiance 0.670 micron]
     1  -0.993  0.385E-01      135   256.00  1.000   [Radiance 0.670 micron]
   

/data/keeling/a/johntl3/miniconda3/envs/at3d_pure_k9/lib/python3.10/site-packages/at3d/solver.py:2337: UserWarning: Number of SHDOM grid cells with optical depth greater than 2: '8'. Max cell optical depth: '2.0755209922790527'
  warnings.warn("Number of SHDOM grid cells with optical depth greater than 2: '{}'. "


  ! Iter Log(Sol)  SplitCrit  Npoints  Nsh(avg)   [Radiance 0.670 micron]
     1  -0.883  0.171E+00      135   256.00  1.000   [Radiance 0.670 micron]
     2  -1.429  0.181E+00      135   256.00  1.000   [Radiance 0.670 micron]
     3  -1.617  0.189E-01      216   256.00  1.000   [Radiance 0.670 micron]
     4  -1.773  0.168E-01      234   256.00  1.000   [Radiance 0.670 micron]
     5  -2.028  0.166E-01      234   256.00  1.000   [Radiance 0.670 micron]
     6  -2.193  0.123E-01      279   256.00  1.000   [Radiance 0.670 micron]
     7  -2.214  0.507E-02      387   256.00  1.000   [Radiance 0.670 micron]
     8  -2.563  0.416E-02      414   256.00  1.000   [Radiance 0.670 micron]
Further cell splitting requires larger array allocations:  Ncells=    1012  Npnts=     578
     9  -2.775  0.272E-02      578   256.00  1.000   [Radiance 0.670 micron]
    10  -2.831  0.273E-02      578   256.00  1.000   [Radiance 0.670 micron]
    11  -3.023  0.274E-02      578   256.00  1.000   [Radiance 0.

/data/keeling/a/johntl3/miniconda3/envs/at3d_pure_k9/lib/python3.10/site-packages/at3d/solver.py:2337: UserWarning: Number of SHDOM grid cells with optical depth greater than 2: '8'. Max cell optical depth: '2.239898920059204'
  warnings.warn("Number of SHDOM grid cells with optical depth greater than 2: '{}'. "


  ! Iter Log(Sol)  SplitCrit  Npoints  Nsh(avg)   [Radiance 0.670 micron]
     1  -0.885  0.182E+00      135   256.00  1.000   [Radiance 0.670 micron]
     2  -1.449  0.192E+00      135   256.00  1.000   [Radiance 0.670 micron]
     3  -1.615  0.207E-01      216   256.00  1.000   [Radiance 0.670 micron]
     4  -1.786  0.185E-01      234   256.00  1.000   [Radiance 0.670 micron]
     5  -2.034  0.180E-01      234   256.00  1.000   [Radiance 0.670 micron]
     6  -2.192  0.126E-01      288   256.00  1.000   [Radiance 0.670 micron]
     7  -2.172  0.572E-02      387   256.00  1.000   [Radiance 0.670 micron]
     8  -2.552  0.461E-02      396   256.00  1.000   [Radiance 0.670 micron]
Further cell splitting requires larger array allocations:  Ncells=    1012  Npnts=     578
     9  -2.773  0.279E-02      578   256.00  1.000   [Radiance 0.670 micron]
    10  -2.836  0.279E-02      578   256.00  1.000   [Radiance 0.670 micron]
    11  -3.004  0.279E-02      578   256.00  1.000   [Radiance 0.

...

  ! Iter Log(Sol)  SplitCrit  Npoints  Nsh(avg)   [Radiance 0.670 micron]
     1  -0.939  0.592E+00      135   256.00  1.000   [Radiance 0.670 micron]
     2  -1.711  0.588E+00      135   256.00  1.000   [Radiance 0.670 micron]
     3  -1.460  0.300E-01      378   256.00  1.000   [Radiance 0.670 micron]
     4  -1.447  0.304E-01      432   256.00  1.000   [Radiance 0.670 micron]
     5  -1.719  0.320E-01      432   256.00  1.000   [Radiance 0.670 micron]
     6  -1.867  0.298E-01      441   256.00  1.000   [Radiance 0.670 micron]
     7  -2.030  0.260E-01      450   256.00  1.000   [Radiance 0.670 micron]
Further cell splitting requires larger array allocations:  Ncells=    1012  Npnts=     578
     8  -2.147  0.183E-01      578   256.00  1.000   [Radiance 0.670 micron]
     9  -2.198  0.186E-01      578   256.00  1.000   [Radiance 0.670 micron]
    10  -2.414  0.187E-01      578   256.00  1.000   [Radiance 0.670 micron]
    11  -2.531  0.190E-01      578   256.00  1.000   [Radiance 0.

/data/keeling/a/johntl3/miniconda3/envs/at3d_pure_k9/lib/python3.10/site-packages/at3d/solver.py:2337: UserWarning: Number of SHDOM grid cells with optical depth greater than 2: '16'. Max cell optical depth: '38.14867401123047'
  warnings.warn("Number of SHDOM grid cells with optical depth greater than 2: '{}'. "


  ! Iter Log(Sol)  SplitCrit  Npoints  Nsh(avg)   [Radiance 0.670 micron]
     1  -0.939  0.593E+00      135   256.00  1.000   [Radiance 0.670 micron]
     2  -1.712  0.590E+00      135   256.00  1.000   [Radiance 0.670 micron]
     3  -1.456  0.301E-01      378   256.00  1.000   [Radiance 0.670 micron]
     4  -1.444  0.304E-01      432   256.00  1.000   [Radiance 0.670 micron]
     5  -1.717  0.318E-01      432   256.00  1.000   [Radiance 0.670 micron]
     6  -1.866  0.332E-01      432   256.00  1.000   [Radiance 0.670 micron]
     7  -2.033  0.264E-01      450   256.00  1.000   [Radiance 0.670 micron]
Further cell splitting requires larger array allocations:  Ncells=    1012  Npnts=     578
     8  -2.145  0.193E-01      578   256.00  1.000   [Radiance 0.670 micron]
     9  -2.166  0.196E-01      578   256.00  1.000   [Radiance 0.670 micron]
    10  -2.387  0.197E-01      578   256.00  1.000   [Radiance 0.670 micron]
    11  -2.496  0.201E-01      578   256.00  1.000   [Radiance 0.

/data/keeling/a/johntl3/miniconda3/envs/at3d_pure_k9/lib/python3.10/site-packages/at3d/solver.py:2337: UserWarning: Number of SHDOM grid cells with optical depth greater than 2: '16'. Max cell optical depth: '41.19316482543945'
  warnings.warn("Number of SHDOM grid cells with optical depth greater than 2: '{}'. "


  ! Iter Log(Sol)  SplitCrit  Npoints  Nsh(avg)   [Radiance 0.670 micron]
     1  -0.939  0.595E+00      135   256.00  1.000   [Radiance 0.670 micron]
     2  -1.712  0.592E+00      135   256.00  1.000   [Radiance 0.670 micron]
     3  -1.452  0.300E-01      378   256.00  1.000   [Radiance 0.670 micron]
     4  -1.442  0.303E-01      432   256.00  1.000   [Radiance 0.670 micron]
     5  -1.715  0.316E-01      432   256.00  1.000   [Radiance 0.670 micron]
     6  -1.866  0.328E-01      432   256.00  1.000   [Radiance 0.670 micron]
     7  -2.035  0.267E-01      450   256.00  1.000   [Radiance 0.670 micron]
Further cell splitting requires larger array allocations:  Ncells=    1012  Npnts=     578
     8  -2.143  0.202E-01      578   256.00  1.000   [Radiance 0.670 micron]
     9  -2.131  0.207E-01      578   256.00  1.000   [Radiance 0.670 micron]
    10  -2.364  0.208E-01      578   256.00  1.000   [Radiance 0.670 micron]
    11  -2.471  0.212E-01      578   256.00  1.000   [Radiance 0.

In [6]:
LUT = xr.Dataset(
    coords = dict(I=Is),
    data_vars = dict(
        tau=(['I'], taus),
        num_unique = (['I'], num_unique),
        diff = (['I'], diff)
    )
)
assert np.all(np.diff(LUT.tau.data) > 0), "LUT tau not monotonic"
assert np.all(np.diff(LUT.I.data) > 0), "LUT I not monotonic"
if(save_LUT):
    LUT.to_netcdf(LUT_outfile)